# [16.4] TokenSHAP and TokenShapley - Solutions

This notebook runs the reference implementation, the visible tests, and direct checks on the committed CUDA verification report.

In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter16_shapley_attribution_baselines"
section = "part4_tokenshap_token_shapley"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_tokenshap_token_shapley.solutions as solutions
import part4_tokenshap_token_shapley.tests as tests

In [ ]:
tests.test_all_coalitions_enumerates_complete_powerset(solutions.all_coalitions)
tests.test_keyword_interaction_token_score_is_target_context_game(
    solutions.keyword_interaction_token_score,
)
tests.test_token_coalition_values_masks_absent_positions(
    solutions.token_coalition_values,
)
tests.test_exact_token_shapley_values_splits_context_target_interaction(
    solutions.exact_token_shapley_values,
)
tests.test_token_baseline_report_checks_efficiency(solutions.token_baseline_report)
tests.test_token_shapley_sampling_report_matches_exact_ranking(
    solutions.token_shapley_sampling_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)

In [ ]:
smoke = solutions.run_smoke_test(cpu=True)
assert smoke["coalitions"]["empty"] == 0.0
assert smoke["coalitions"]["target_only"] == 1.0
assert smoke["coalitions"]["context_and_target"] == 3.0
assert smoke["exact"]["exact_values"] == [0.0, 1.0, 0.0, 2.0]
assert smoke["exact"]["baseline"]["satisfies_efficiency"]
assert smoke["sampled"]["approximates_exact"]
assert smoke["sampled"]["sampled_top_token"] == "Paris"
smoke

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]

assert report["accepted"] and report["tests_passed"]
assert report["gt_tier"] == "GT-0"
assert report["notebook_id"] == "16_4_tokenshap_and_tokenshapley"
assert gpu["cuda_available"] and gpu["preflight_passed"]
assert gpu["model_family"] == "cuda_trained_tiny_token_scorer_mlp"
assert gpu["token_count"] == 4
assert gpu["coalition_count"] == 16
assert gpu["training_example_count"] == 16
assert gpu["training_steps"] == 1200
assert gpu["fit_mse"] <= 1e-10
assert gpu["fit_max_abs_error"] <= 1e-5
assert gpu["exact_shapley_max_abs_error"] <= 1e-5
assert gpu["sampled_max_abs_error"] <= 0.1
assert gpu["sampled_rank_matches"]
assert gpu["top_token"] == "Paris"
assert gpu["sampled_top_token"] == "Paris"
assert gpu["baseline_efficiency_error"] <= 1e-8
assert gpu["satisfies_efficiency"]
assert gpu["shuffled_control_error"] >= 1.0
assert gpu["shuffled_control_rejected"]
assert gpu["peak_vram_gb"] < 1.0
assert gpu["within_vram_budget"]

tests.test_committed_gpu_report_matches_token_shapley_contract(gpu)
{
    "device": gpu["device"],
    "exact_shapley_max_abs_error": gpu["exact_shapley_max_abs_error"],
    "sampled_max_abs_error": gpu["sampled_max_abs_error"],
    "shuffled_control_error": gpu["shuffled_control_error"],
    "peak_vram_gb": gpu["peak_vram_gb"],
}